# Supply Chain Event Streaming - Producer

Previously, we built a static supply chain graph in Neo4j using Apache Spark. Here we add a streaming layer - shipment events flow through Confluent Cloud Kafka in real-time and land in Neo4j as enriched graph properties.

This notebook is the **producer**. It generates synthetic shipment status events and publishes them to a Confluent Cloud Kafka topic. Run the consumer notebook alongside this one to see events land in Neo4j in real-time.

## Architecture

```
+---------------------------------------------------------+
|                    Confluent Cloud                      |
|                                                         |
|  +------------------+          +---------------------+  |
|  |  Producer        |--------->|    Kafka Topic      |  |
|  |  (this notebook) |          |  shipment-events    |  |
|  +------------------+          +---------------------+  |
|                                                         |
+---------------------------------------------------------+
```

## Before Running This Notebook

You will need a Confluent Cloud account with a running cluster. Export the following environment variables in your shell before starting Jupyter:

```bash
export CONFLUENT_BOOTSTRAP_SERVERS=your_cluster.confluent.cloud:9092
export CONFLUENT_API_KEY=your_api_key
export CONFLUENT_API_SECRET=your_api_secret
```

You will also need to create the topic `shipment-events` in the Confluent Cloud UI before running this notebook. Use the default settings with 1 partition.

## 1. Install Dependencies

All versions are pinned for reproducibility. Run this cell once.

In [1]:
%pip install "confluent-kafka==2.15.0" \
             "faker==40.36.0" \
             "numpy==2.2.6" --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import numpy as np
import os
import time
import uuid

from confluent_kafka import Producer
from faker import Faker
from IPython.display import clear_output

## 3. Configuration

Connection details are read from environment variables - no credentials are hard-coded in the notebook.

In [3]:
BOOTSTRAP_SERVERS = os.environ["CONFLUENT_BOOTSTRAP_SERVERS"]
API_KEY           = os.environ["CONFLUENT_API_KEY"]
API_SECRET        = os.environ["CONFLUENT_API_SECRET"]
TOPIC             = "shipment-events"
RANDOM_SEED       = 42

print("Credentials set.")

Credentials set.


## 4. Create the Kafka Producer

We configure the producer with SASL/SSL authentication, which is required
for all Confluent Cloud clusters.

In [4]:
producer = Producer({
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms":   "PLAIN",
    "sasl.username":     API_KEY,
    "sasl.password":     API_SECRET,
    "log_level":         0,
})

def delivery_report(err, msg):
    """Callback fired when a message is delivered or fails."""
    if err is not None:
        print(f"Delivery failed: {err}")

print("Producer created.")

Producer created.


## 5. Define the Supply Chain Data

We reuse the same supply chain structure - the same node IDs so events land on nodes that already exist in Neo4j. A fixed random seed guarantees the IDs match across both notebooks.

Each shipment event carries:

- `shipment_id` - unique identifier for this shipment
- `supplier_id` - the originating supplier
- `warehouse_id` - the warehouse handling the shipment
- `dist_center_id` - the distribution center receiving it
- `retailer_id` - the final destination
- `status` - current status: `departed`, `in_transit`, `delayed` or `delivered`
- `timestamp` - when this status update occurred
- `delay_minutes` - how many minutes delayed (0 if on time)

In [5]:
rng  = np.random.default_rng(RANDOM_SEED)
fake = Faker()
Faker.seed(RANDOM_SEED)

# Match node IDs from the Part 1 notebook
SUPPLIER_IDS    = [f"S{i:03d}" for i in range(20)]
WAREHOUSE_IDS   = [f"W{i:03d}" for i in range(12)]
DIST_CENTER_IDS = [f"DC{i:03d}" for i in range(10)]
RETAILER_IDS    = [f"R{i:03d}" for i in range(30)]

STATUSES        = ["departed", "in_transit", "delayed", "delivered"]
STATUS_WEIGHTS  = [0.25, 0.45, 0.15, 0.15]   # 15% of shipments are delayed

def make_shipment_event():
    """Generate a single synthetic shipment status event."""
    status = str(rng.choice(STATUSES, p = STATUS_WEIGHTS))
    return {
        "shipment_id":    str(uuid.uuid4()),
        "supplier_id":    str(rng.choice(SUPPLIER_IDS)),
        "warehouse_id":   str(rng.choice(WAREHOUSE_IDS)),
        "dist_center_id": str(rng.choice(DIST_CENTER_IDS)),
        "retailer_id":    str(rng.choice(RETAILER_IDS)),
        "status":         status,
        "timestamp":      time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "delay_minutes":  int(rng.integers(15, 240)) if status == "delayed" else 0,
    }

# Preview a sample event
sample = make_shipment_event()
print("Sample event:")
print(json.dumps(sample, indent = 2))

Sample event:
{
  "shipment_id": "be512642-05ef-47df-947f-9833ece99ccd",
  "supplier_id": "S013",
  "warehouse_id": "W005",
  "dist_center_id": "DC004",
  "retailer_id": "R025",
  "status": "delayed",
  "timestamp": "2026-08-07T21:17:35Z",
  "delay_minutes": 34
}


## 6. Publish Shipment Events to Kafka

We publish events to the `shipment-events` topic with a short pause between
each one to simulate a real-time stream. Set `N_EVENTS = -1` to stream
continuously until interrupted. The display is refreshed every `PRINT_EVERY`
events to keep the cell output manageable.

Start the consumer or dashboard notebook now and you will see events
arriving in Neo4j as this cell runs.

In [6]:
N_EVENTS      = -1    # set to -1 to stream continuously
DELAY_SECONDS = 0.5   # pause between events
PRINT_EVERY   = 20    # refresh display every N events

def _produce_single_event(counter, counts):
    """Generate and publish one shipment event."""
    event = make_shipment_event()
    producer.produce(
        topic    = TOPIC,
        key      = event["shipment_id"],
        value    = json.dumps(event),
        callback = delivery_report,
    )
    producer.poll(0)
    counts[event["status"]] += 1
    return event

def produce_events(num_events = 50, delay_seconds = 0.5, print_every = 20):
    """Publish events to Kafka. Set num_events = -1 to run continuously."""
    counter = 0
    counts  = {s: 0 for s in STATUSES}
    label   = "endlessly" if num_events == -1 else str(num_events)
    print(f"Publishing {label} events to topic '{TOPIC}'...")
    print(f"Delay between events: {delay_seconds}s | Refresh every: {print_every} events")

    try:
        while True:
            event    = _produce_single_event(counter, counts)
            counter += 1

            if counter % print_every == 0 or counter == 1:
                clear_output(wait = True)
                print(f"Published {counter} events to topic '{TOPIC}'")
                print(f"Delay: {delay_seconds}s | Refresh every: {print_every}\n")
                print("Latest event:")
                icon = "!!" if event["status"] == "delayed" else "->"
                print(f"  {icon} {event['shipment_id'][:8]}... "
                      f"{event['supplier_id']} -> {event['warehouse_id']} -> "
                      f"{event['dist_center_id']} -> {event['retailer_id']} "
                      f"[{event['status']}]")
                print("\nStatus breakdown:")
                for status, count in counts.items():
                    print(f"  {status:<12} : {count}")

            if num_events != -1 and counter >= num_events:
                break

            time.sleep(delay_seconds)

    except KeyboardInterrupt:
        print("\nProducer stopped by user.")
    finally:
        producer.flush(timeout = 10)
        print(f"Finished. Total events published: {counter}")

produce_events(
    num_events    = N_EVENTS,
    delay_seconds = DELAY_SECONDS,
    print_every   = PRINT_EVERY,
)

Published 500 events to topic 'shipment-events'
Delay: 0.5s | Refresh every: 20

Latest event:
  -> 756235e6... S004 -> W005 -> DC003 -> R011 [departed]

Status breakdown:
  departed     : 132
  in_transit   : 209
  delayed      : 85
  delivered    : 74

Producer stopped by user.
Finished. Total events published: 502


## 7. Verify Delivery

Check the Confluent Cloud UI under your cluster's Topics section to confirm messages have landed on the `shipment-events` topic. You should see the message count and throughput graphs updating.

Switch to the dashboard notebook to see the events being written into Neo4j.